In [ ]:
!pip install -q transformers accelerate bitsandbytes torch gtts --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.6 MB/s eta 0:00:00


In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoModelForVision2Seq, AutoTokenizer, AutoProcessor

In [ ]:
### This code is running on Google Colab T4 GPU

#### Qwen 2.5 for Image Caption

In [ ]:
qwen_model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

qwen_model = AutoModelForVision2Seq.from_pretrained(
    qwen_model_id,
    dtype=torch.float16,
    device_map="auto"
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
qwen_processor = AutoProcessor.from_pretrained(qwen_model_id)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2242: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
long_prompt = """
You are a compassionate storyteller. Using the attached photo, craft a 120–180 word micro‑story intended for an older adult and their family. The story should gently evoke memories, spark warm conversation, and support emotional well‑being.

Guidelines:
- Focus on mood, place, season, and relationships; avoid listing objects.
- Weave in 2–3 sensory details (sounds, scents, textures, light).
- Use warm, respectful language and short, vivid sentences.
- Avoid definitive claims about names, ages, or locations. Use gentle, tentative phrasing (perhaps, it seems, maybe).
- If people appear, emphasize connection and small rituals rather than appearance.
- Be inclusive and avoid stereotypes; balance nostalgia with quiet hope.
- If text is clearly legible in the image, you may thoughtfully incorporate it.
- If the scene is ambiguous, lean into universal themes (family, gatherings, journeys, everyday moments).

Output:
- 1–2 paragraphs of story.

Variants (pick one voice if you want to steer style):
- Voice A (third‑person close): Tell the story from a gentle narrator’s view.
- Voice B (first‑person elder): Write as if an older adult is recalling the moment in the photo.
- Voice C (second person): Address a loved one directly, with tenderness and gratitude.

Examples of style knobs you can add:
- Tone: warm and hopeful; lightly bittersweet; playful nostalgia.
- Era cues: hint at a decade only if strongly suggested by the image.
- Cultural touch: include respectful, non‑stereotyped details only if clearly present.

"""

In [ ]:
short_prompt = """Write a 120–180 word micro‑story inspired by this photo for an older adult and their family. Describe the photo honestly, write the story to evoke gentle reminiscence and well‑being. Use warm, simple language, 2–3 sensory details, and avoid object lists. Use tentative phrasing for uncertain facts. Emphasize connection and small rituals."""


In [ ]:
# Both long and short prompt works well. Use long one to generate story with variants (different perspective), use short one to generate clean and short story.
# I also tried two outputs of generating 1. 1–2 paragraphs of story 2. Then add “Conversation starters:” followed by two open‑ended, gentle questions that invite sharing (e.g., “What songs did you hear at gatherings like this?”).
# However, generating two open-ended questions that invite sharing (to echo with our proposal of sparking meaningful family conversation) did not work well (sometimes it is off topic or too general) so I removed it


In [ ]:
def generate_story_qwen(image_path, prompt=short_prompt):
    image = Image.open(image_path).convert("RGB")

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]

    text_prompt = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_processor(text=[text_prompt], images=[image], return_tensors="pt").to("cuda")

    output_ids = qwen_model.generate(**inputs, max_new_tokens=300)

    generated_text = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

In [ ]:
image_path = "example_photo/img_1.jpg" # should later replace with input
# three examples for testing ["img_1.jpg", "img_2.jpg", "img_3.png"]
image_story = generate_story_qwen(image_path)
### usually takes about 2 mins to generate the result on T4 GPU

print(image_story)

------ Qwen2.5-VL Story ------
In the soft light of a late afternoon, a family stands together on a bridge, their laughter mingling with the gentle breeze. The city skyline stretches behind them, a tapestry of towering skyscrapers and twinkling lights. The woman, her arm around the child, points towards something in the distance, perhaps a familiar landmark or a new adventure. The man beside her smiles warmly, his hand gently resting on the child's shoulder. The scene is one of simple joy and shared moments, a snapshot of life's little treasures. As they look out over the water, the family seems to be building memories that will last a lifetime, each moment a testament to the love and connection that binds them.


#### Translate from English to Chinese

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

In [ ]:
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

def translate_to_chinese(text):
    tokenizer.src_lang = "en"
    encoded = tokenizer(text, return_tensors="pt").to(model.device)
    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id("zh"))
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)


In [ ]:
cantonese_text = translate_to_chinese(image_story)
print("Translated text:", cantonese_text)

Translated text: 在一个晚上的温柔光明中,一个家庭站在桥上,他们的笑声与温柔的气息混合在一起。 城市的天花板在他们背后伸展,一块旋转的天花板和旋转的灯光。 妇女,她的手在孩子周围,指向距离,也许一个熟悉的景点或一个新的冒险。 旁边的男人微笑温暖,他的手轻轻地休息在孩子的肩膀上。 场景是简单的快乐和共享的时刻之一,生活的小宝藏的瞬间。 他们看着水上,家庭似乎正在建造记忆,将持续一生,每一刻都是爱情和连接的遗嘱。
